# 01 — A\* Path Planning on an Occupancy Grid

**Section:** Motion Planning · **Mirrors MATLAB:** *Motion Planners (RRT, PRM, Hybrid A\*)*

This notebook implements **A\*** search on a 2-D occupancy grid with 8-connected motion. A\* expands grid cells in order of `f = g + h`, where `g` is the cost-to-come and `h` is the heuristic estimate of cost-to-go.

We use the **Euclidean distance** heuristic, which is admissible (never overestimates) for 8-connected grids with unit-or-√2 edge costs.


## Intuition — what's actually going on?

Imagine you have a top-down view of a maze and you want the shortest path from room A to room B. The naive thing to do is explore *every* room — that works, but it's slow. A better idea: at every step, prefer to explore the room that looks closest to the goal. That's the whole trick behind A\*.

Each room (cell in our grid) gets two numbers:

- **`g`** — how far you've actually walked from the start to get here.
- **`h`** — how far you *think* you still have to go (a guess, like straight-line distance).

A\* always picks next the room whose `f = g + h` is smallest — i.e. the cheapest *total trip* so far. As long as your guess `h` satisfies a slightly stronger property called **consistency** (defined precisely below; for graph search with a closed list, plain admissibility is not enough), A\* is guaranteed to return the genuinely shortest path the first time it pops the goal from the queue.

Mathematically A\* is dynamic programming with a domain-knowledge prior — same as Dijkstra (notebook 13) but with the heuristic $h$ biasing exploration toward the goal. The math below proves this via Bellman's Principle of Optimality.


## Analytical derivation

**Problem.** Given a graph $G = (V, E)$ with non-negative edge weights $c: E \to \mathbb{R}_{\ge 0}$, a start node $s$, and a goal node $t$, find $\pi = (s = v_0, v_1, \ldots, v_k = t)$ minimizing $\sum_{i=1}^k c(v_{i-1}, v_i)$.

**Best-first search.** For every visited node $n$ keep $g(n)$ = actual cost of the best known path from $s$ to $n$. A heuristic $h: V \to \mathbb{R}_{\ge 0}$ estimates the remaining cost from $n$ to $t$. A\* always expands the node with smallest

$$f(n) = g(n) + h(n).$$

**Admissibility.** $h$ is *admissible* iff $h(n) \le h^*(n)$ for every $n$, where $h^*(n)$ is the true cost-to-go from $n$ to $t$. In particular $h^*(s) = d(s, t)$, the optimal start-to-goal cost. Euclidean distance is admissible for our 8-connected grid because no path can be shorter than the straight line.

**Consistency.** $h$ is *consistent* iff $h(n) \le c(n, n') + h(n')$ for every edge. Consistency implies admissibility, and Euclidean distance is consistent.

**Optimality theorem.** Assume edge weights $c \ge 0$, $h \ge 0$, $h(t) = 0$ (which follows from admissibility plus $h \ge 0$ and $h^*(t) = 0$), and $h$ consistent. Then A\* expands nodes in order of non-decreasing $f$, and the first time $t$ is popped from the open set, $g(t) = d(s, t)$.

### Compatibility check — math ↔ code

| Step | Code |
|---|---|
| $h(n) = \sqrt{(\Delta x)^2 + (\Delta y)^2}$ | `def h(a, b): return np.hypot(a[0]-b[0], a[1]-b[1])` |
| Edge cost = step distance ($1$ or $\sqrt 2$) | `tent = gv + np.hypot(dy, dx)` |
| Always pop smallest $f$ | `heapq.heappop(open_heap)` on `(f, g, node, parent)` |
| Push neighbor with $f = g + h$ | `heapq.heappush(open_heap, (tent + h((ny,nx), goal), tent, (ny,nx), cur))` |
| Skip closed nodes | `if cur in came: continue` |
| Optimal path on first pop of goal | `if cur == goal: return path, ...` |
| Reconstruct path by parent pointers | `while cur is not None: path.append(cur); cur = came[cur]` |
| Tie-break on $f$: by $g$, then lexicographic on cell | Python tuple ordering of `(f, g, node, parent)` in heap entries |

Tie-breaking note: any consistent tie-break gives an optimal path; *preferring larger* $g$ (negate it in the key) often reduces expansions in symmetric grids.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

np.random.seed(42)


In [ ]:
# Build a 30 x 50 occupancy grid with random rectangular obstacles
H, W = 30, 50
grid = np.zeros((H, W), dtype=np.uint8)
for _ in range(40):
    y, x = np.random.randint(2, H - 3), np.random.randint(2, W - 3)
    grid[y - 1:y + 2, x - 1:x + 2] = 1

start = (2, 2)
goal = (H - 3, W - 3)
grid[start] = 0
grid[goal] = 0
print(f"Grid: {H}x{W}, {int(grid.sum())} obstacle cells")


In [ ]:
def astar(grid, start, goal):
    H, W = grid.shape

    def h(a, b):
        return np.hypot(a[0] - b[0], a[1] - b[1])

    open_heap = [(h(start, goal), 0.0, start, None)]
    came_from = {}
    g_score = {start: 0.0}
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1),
            (-1, -1), (-1, 1), (1, -1), (1, 1)]

    while open_heap:
        f, g, current, parent = heapq.heappop(open_heap)
        if current in came_from:
            continue
        came_from[current] = parent
        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            # Return path, plus came_from (= closed/expanded set) and g_score
            # (= discovered set). g_score is debug instrumentation only;
            # production callers should drop it. came_from is needed for
            # accurate "expanded N nodes" reporting (Bertsekas council fix).
            return path[::-1], came_from, g_score
        for dy, dx in nbrs:
            ny, nx = current[0] + dy, current[1] + dx
            if not (0 <= ny < H and 0 <= nx < W) or grid[ny, nx]:
                continue
            tentative = g + np.hypot(dy, dx)
            neighbor = (ny, nx)
            if tentative < g_score.get(neighbor, np.inf):
                g_score[neighbor] = tentative
                heapq.heappush(
                    open_heap,
                    (tentative + h(neighbor, goal), tentative, neighbor, current),
                )
    return None, came_from, g_score


path, closed, g_score = astar(grid, start, goal)
print(f"Path length: {len(path)} cells  ·  expanded {len(closed)} nodes  ·  discovered {len(g_score)} nodes")


In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(grid, cmap='Greys', origin='lower')

# Heat-map of CLOSED nodes (i.e., popped/expanded). Using `closed` (came_from)
# not `g_score` matches the "expanded N nodes" count in the print above.
expanded_map = np.full_like(grid, np.nan, dtype=float)
for (y, x) in closed:
    expanded_map[y, x] = g_score[(y, x)]
ax.imshow(expanded_map, cmap='viridis', alpha=0.4, origin='lower')

ys, xs = zip(*path)
ax.plot(xs, ys, 'r-', lw=2.5, label='A* path')
ax.plot(start[1], start[0], 'go', markersize=14, label='Start')
ax.plot(goal[1], goal[0], 'b*', markersize=18, label='Goal')
ax.legend(loc='upper right')
ax.set_title('A* Path Planning — expanded cells shaded by g-score')
plt.tight_layout()
plt.show()


## Extensions

- **D\* / D\* Lite** — incremental replanning when the grid changes (use for dynamic environments).
- **Hybrid A\*** — extends A\* to continuous heading + kinematic constraints (used for car-like robots).
- **Jump Point Search** — a symmetry-breaking optimization for uniform grids; often 10×+ faster than A\*.


## References & rigor notes

**Complexity.** With a binary-heap priority queue, A\* runs in $O(|E| \log |V|)$ time using either decrease-key (heap holds at most $|V|$ entries) or lazy deletion (heap can hold up to $|E|$ entries; both give the same asymptotic bound since $\log |E| = \Theta(\log |V|)$ for a graph). With a Fibonacci heap: $O(|E| + |V| \log |V|)$. Space: $O(|E|)$ for the heap (lazy-deletion case) plus $O(|V|)$ for the $g$-score and parent maps.

**Theorem** (Optimality under consistent heuristic; Hart, Nilsson & Raphael, 1968).
*Assume edge weights $c \ge 0$, heuristic $h \ge 0$ with $h(t) = 0$, and $h$ consistent. Then A\* expands nodes in non-decreasing order of $f$, and the first time the goal $t$ is popped from the open set, $g(t) = d(s, t)$ (optimal cost).*

*Proof sketch.* First show $f$ is non-decreasing along any path from $s$. For any edge $(n_i, n_{i+1})$:
$$f(n_{i+1}) = g(n_i) + c(n_i, n_{i+1}) + h(n_{i+1}) \ge g(n_i) + h(n_i) = f(n_i)$$
by consistency. Hence $f$ is non-decreasing along any path.

Now suppose at the moment $t$ is popped that a strictly cheaper path $\pi'$ to $t$ exists with cost $g'(t) < g(t)$. Then some node $n'$ on $\pi'$ is still in the open heap with $g$-value matching its prefix on $\pi'$, and $f(n') \le g'(t) < g(t) = f(t)$ (using $h(t) = 0$). But A\* popped $t$, so $f(t) \le f(n')$ — contradiction. Hence $g(t)$ is optimal. ∎

This is a consequence of the **Principle of Optimality** (Bellman, 1957): sub-paths of optimal paths are optimal, hence label-correcting algorithms that maintain $g(n)$ as a tight upper bound and expand in $f$-order discover the optimal $g(n)$ at first pop.

**References.**
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). *A formal basis for the heuristic determination of minimum cost paths*. IEEE Transactions on Systems Science and Cybernetics, 4(2), 100-107.
- Bellman, R. (1957). *Dynamic Programming*. Princeton University Press.
- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach*, 4th ed., Pearson, ch. 3.
